# Q-SITE 2024: IBM Quantum Hackathon

## Lab 2: Quantum Enigma - Four-square chessboard



This challenge is based on the Quantum Enigmas produced by Institut Quantique at the Université de Sherbrooke. Watch this video to see the details of the Quantum Enigma entitled ***The Four-Square Chessboard***: https://youtu.be/UuVbtFXOEKQ?feature=shared .

In this challenge, you will code the solution to the enigma using Qiskit.

The code required to complete the exercise should be typed under the line which has the `## WRITE YOUR CODE BELOW HERE ##` comment. Avoid making changes to any provided code as this could alter how other code in the notebook functions. You will submit answers to the grader methods that are called in this notebook. This lab has 4 graded exercises.

The Four-Square Chessboard enigma is summarized as follows. Alice and Bob play a game in which Bob must find the location of a key hidden under one of the squares of a chessboard. Each chessboard square also has a coin with one side labeled 0 and the other side labeled 1, and each coin is placed randomly on one of these two sides. Alice must indicate the location of the hidden key to Bob by only flipping one coin on the board. Bob is provided with no information except the state of the chessboard after Alice flips the coin (i.e. he does not know which coin Alice flipped).

Below is an example of the chessboard with the coin values shown in the video, with the example location of the key in red. However, the board could be in any configuration. In this case, there are 64 possible configurations, i.e. 16 possible coin configurations $\times$ 4 possible key locations.

<center>
<table border=1>
    <colgroup>
        <col span="1" style="width:50px; text-align: center;">
        <col span="1" style="width:50px; text-align: center;">
    </colgroup>
    <tbody>
        <tr style="height:50px"><td>0</td><td bgcolor='red'>1</span></td></tr>
        <tr style="height:50px"><td>1</td><td>0</td></tr>
    </tbody>
</table>
</center>

Alice and Bob come into the game with a strategy to allow Bob to find the key given the rules of the game. It is based on computing parities of the coin values on the board. Alice will flip a coin based on her calculation of the partities of the key location expressed in binary, and a focus square location that is chosen by computing the parities of coins on the board. Bob will repeat this parity calculation of the coins on the board in order to find the location of the key.

To validate this strategy, we can use a quantum computer to represent all of the possible configurations simultaneously and compute this strategy. 

In [1]:
import sys
import os
sys.path.append(os.path.abspath('D:/jupynote/Qsite/Q-SITE-IBM-Coding-Challenge/grader'))

In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorSampler

from qsite24_ibm_grader import *

### Part 1: Construct circuit for four-square chessboard

Use this initial code that sets up the quantum circuit with a set of quantum and classical registers. This setup is a bit different from video where the qubits were not divided into named register and were instead specified solely by their index number. This may become cumbersome when scaling up the circuit as you will do in Part 2. Register allow your code to be more readable.

The registers below represent the following:

- `q_reg_squares`: represents the squares of the chessboard. In the binary mapping of the square numbers, the first bit represents the row of the square while the second bit represents the column of the square.
- `q_reg_key`: the key location in binary.
- `q_reg_afocus`: Alice's computed focus square in binary.
- `q_reg_bfocus`: Bob's computed focus square in binary.
- `c_reg_key_bfocus`: The measured key location (first pair of qubits) and Bob's focus square (second pair of qubits). These are measured into the same classical register for convenience to clearly see the result in the end.

In [3]:
q_reg_squares = QuantumRegister(4, name='squares')
q_reg_key = QuantumRegister(2, name='key')
q_reg_afocus = QuantumRegister(2, name='afocus')
q_reg_bfocus = QuantumRegister(2, name='bfocus')

c_reg_key_bfocus = ClassicalRegister(4, name='c_key_bfocus')

qc = QuantumCircuit(q_reg_squares, q_reg_key, q_reg_afocus, q_reg_bfocus, c_reg_key_bfocus)
qc.draw()

squares_0: 
                
     squares_1: 
                
     squares_2: 
                
     squares_3: 
                
         key_0: 
                
         key_1: 
                
      afocus_0: 
                
      afocus_1: 
                
      bfocus_0: 
                
      bfocus_1: 
                
c_key_bfocus: 4/

#### Step 1

In the first step, place the qubits of the square and key registers into equal superpositions of 0 and 1. This will effectively represent all possible combinations of coin values and key locations.

In [4]:
## WRITE YOUR CODE BELOW HERE ##
qc.h(q_reg_key)
qc.h(q_reg_squares)
qc.draw()

┌───┐
     squares_0: ┤ H ├
                ├───┤
     squares_1: ┤ H ├
                ├───┤
     squares_2: ┤ H ├
                ├───┤
     squares_3: ┤ H ├
                ├───┤
         key_0: ┤ H ├
                ├───┤
         key_1: ┤ H ├
                └───┘
      afocus_0: ─────
                     
      afocus_1: ─────
                     
      bfocus_0: ─────
                     
      bfocus_1: ─────
                     
c_key_bfocus: 4/═════

#### Step 2

In the next step, the focus square is selected by computing the partities of the bottom row and right-most column.

In [5]:
## WRITE YOUR CODE BELOW HERE ##
qc.cx(q_reg_squares[1], q_reg_afocus[0])
qc.cx(q_reg_squares[3], q_reg_afocus[0])
qc.cx(q_reg_squares[2], q_reg_afocus[1])
qc.cx(q_reg_squares[3], q_reg_afocus[1])

qc.draw()

┌───┐                    
     squares_0: ┤ H ├────────────────────
                ├───┤                    
     squares_1: ┤ H ├──■─────────────────
                ├───┤  │                 
     squares_2: ┤ H ├──┼────■────────────
                ├───┤  │    │            
     squares_3: ┤ H ├──┼────┼────■────■──
                ├───┤  │    │    │    │  
         key_0: ┤ H ├──┼────┼────┼────┼──
                ├───┤  │    │    │    │  
         key_1: ┤ H ├──┼────┼────┼────┼──
                └───┘┌─┴─┐  │  ┌─┴─┐  │  
      afocus_0: ─────┤ X ├──┼──┤ X ├──┼──
                     └───┘┌─┴─┐└───┘┌─┴─┐
      afocus_1: ──────────┤ X ├─────┤ X ├
                          └───┘     └───┘
      bfocus_0: ─────────────────────────
                                         
      bfocus_1: ─────────────────────────
                                         
c_key_bfocus: 4/═════════════════════════

#### Step 3

Now Alice must determine which coin to flip. This is done by computing the location of the square by computing the parities of each bit between the key location and the focus square registers. When done, the square with the coin to flip will be stored in Alice's focus register.

In [6]:
## WRITE YOUR CODE BELOW HERE ##

qc.cx(q_reg_key[0], q_reg_afocus[0])
qc.cx(q_reg_key[1], q_reg_afocus[1])

qc.draw()

┌───┐                              
     squares_0: ┤ H ├──────────────────────────────
                ├───┤                              
     squares_1: ┤ H ├──■───────────────────────────
                ├───┤  │                           
     squares_2: ┤ H ├──┼────■──────────────────────
                ├───┤  │    │                      
     squares_3: ┤ H ├──┼────┼────■────■────────────
                ├───┤  │    │    │    │            
         key_0: ┤ H ├──┼────┼────┼────┼────■───────
                ├───┤  │    │    │    │    │       
         key_1: ┤ H ├──┼────┼────┼────┼────┼────■──
                └───┘┌─┴─┐  │  ┌─┴─┐  │  ┌─┴─┐  │  
      afocus_0: ─────┤ X ├──┼──┤ X ├──┼──┤ X ├──┼──
                     └───┘┌─┴─┐└───┘┌─┴─┐└───┘┌─┴─┐
      afocus_1: ──────────┤ X ├─────┤ X ├─────┤ X ├
                          └───┘     └───┘     └───┘
      bfocus_0: ───────────────────────────────────
                                                   
      bfocus_1: ───────────────────────────────────
                                                   
c_key_bfocus: 4/═══════════════════════════════════

In [7]:
## SUBMIT CIRCUIT TO GRADER
qsite24_grader_lab2ex1a(qc)

Congratulations! 🎉 Your answer is correct.


#### Step 4

Now flip the coins according to the value in Alice's focus register.

In [8]:
## WRITE YOUR CODE BELOW HERE ##

qc.ccx(q_reg_afocus[0], q_reg_afocus[1], q_reg_squares[3])

qc.x(q_reg_afocus[0])
qc.ccx(q_reg_afocus[0], q_reg_afocus[1], q_reg_squares[2])
qc.x(q_reg_afocus[0])

qc.x(q_reg_afocus[1])
qc.ccx(q_reg_afocus[0], q_reg_afocus[1], q_reg_squares[1])
qc.x(q_reg_afocus[1])

qc.x(q_reg_afocus[0])
qc.x(q_reg_afocus[1])
qc.ccx(q_reg_afocus[0], q_reg_afocus[1], q_reg_squares[0])
qc.x(q_reg_afocus[1])
qc.x(q_reg_afocus[0])

qc.draw()

┌───┐                                                       »
     squares_0: ┤ H ├───────────────────────────────────────────────────────»
                ├───┤                                                  ┌───┐»
     squares_1: ┤ H ├──■───────────────────────────────────────────────┤ X ├»
                ├───┤  │                                     ┌───┐     └─┬─┘»
     squares_2: ┤ H ├──┼────■────────────────────────────────┤ X ├───────┼──»
                ├───┤  │    │                      ┌───┐     └─┬─┘       │  »
     squares_3: ┤ H ├──┼────┼────■────■────────────┤ X ├───────┼─────────┼──»
                ├───┤  │    │    │    │            └─┬─┘       │         │  »
         key_0: ┤ H ├──┼────┼────┼────┼────■─────────┼─────────┼─────────┼──»
                ├───┤  │    │    │    │    │         │         │         │  »
         key_1: ┤ H ├──┼────┼────┼────┼────┼────■────┼─────────┼─────────┼──»
                └───┘┌─┴─┐  │  ┌─┴─┐  │  ┌─┴─┐  │    │  ┌───┐  │  ┌───┐  │  »
      afocus_0: ─────┤ X ├──┼──┤ X ├──┼──┤ X ├──┼────■──┤ X ├──■──┤ X ├──■──»
                     └───┘┌─┴─┐└───┘┌─┴─┐└───┘┌─┴─┐  │  └───┘  │  ├───┤  │  »
      afocus_1: ──────────┤ X ├─────┤ X ├─────┤ X ├──■─────────■──┤ X ├──■──»
                          └───┘     └───┘     └───┘               └───┘     »
      bfocus_0: ────────────────────────────────────────────────────────────»
                                                                            »
      bfocus_1: ────────────────────────────────────────────────────────────»
                                                                            »
c_key_bfocus: 4/════════════════════════════════════════════════════════════»
                                                                            »
«                          ┌───┐     
«     squares_0: ──────────┤ X ├─────
«                          └─┬─┘     
«     squares_1: ────────────┼───────
«                            │       
«     squares_2: ────────────┼───────
«                            │       
«     squares_3: ────────────┼───────
«                            │       
«         key_0: ────────────┼───────
«                            │       
«         key_1: ────────────┼───────
«                ┌───┐       │  ┌───┐
«      afocus_0: ┤ X ├───────■──┤ X ├
«                ├───┤┌───┐  │  ├───┤
«      afocus_1: ┤ X ├┤ X ├──■──┤ X ├
«                └───┘└───┘     └───┘
«      bfocus_0: ────────────────────
«                                    
«      bfocus_1: ────────────────────
«                                    
«c_key_bfocus: 4/════════════════════
«

In [9]:
## SUBMIT CIRCUIT TO GRADER
qsite24_grader_lab2ex1b(qc)

Congratulations! 🎉 Your answer is correct.


#### Step 5

Now it's Bob's turn. To determine the coin location, compute the parity of the last row and column just as Alice did previously. Bob's computation of the key location should be stored in his focus register. Once this is done, measure out the key register and Bob's focus register into the same classical register (this code is given).

In [10]:
## WRITE YOUR CODE BELOW HERE ##
qc
qc.cx(q_reg_squares[1], q_reg_bfocus[0])
qc.cx(q_reg_squares[3], q_reg_bfocus[0])
qc.cx(q_reg_squares[2], q_reg_bfocus[1])
qc.cx(q_reg_squares[3], q_reg_bfocus[1])
# 获取 q_reg_key 和 q_reg_bfocus 中所有量子比特
all_qubits = q_reg_key[:] + q_reg_bfocus[:]

# 对这些量子比特进行测量，结果存入 c_reg_key_bfocus
qc.measure(all_qubits, c_reg_key_bfocus)


qc.draw()


┌───┐                                                        »
     squares_0: ┤ H ├────────────────────────────────────────────────────────»
                ├───┤                                                        »
     squares_1: ┤ H ├──■─────────────────────────────────────────────────────»
                ├───┤  │                                           ┌───┐     »
     squares_2: ┤ H ├──┼────■──────────────────────────────────────┤ X ├─────»
                ├───┤  │    │                         ┌───┐        └─┬─┘     »
     squares_3: ┤ H ├──┼────┼────■────■───────────────┤ X ├──────────┼───────»
                ├───┤  │    │    │    │            ┌─┐└─┬─┘          │       »
         key_0: ┤ H ├──┼────┼────┼────┼────■───────┤M├──┼────────────┼───────»
                ├───┤  │    │    │    │    │       └╥┘  │  ┌─┐       │       »
         key_1: ┤ H ├──┼────┼────┼────┼────┼────■───╫───┼──┤M├───────┼───────»
                └───┘┌─┴─┐  │  ┌─┴─┐  │  ┌─┴─┐  │   ║   │  └╥┘┌───┐  │  ┌───┐»
      afocus_0: ─────┤ X ├──┼──┤ X ├──┼──┤ X ├──┼───╫───■───╫─┤ X ├──■──┤ X ├»
                     └───┘┌─┴─┐└───┘┌─┴─┐└───┘┌─┴─┐ ║   │   ║ └───┘  │  ├───┤»
      afocus_1: ──────────┤ X ├─────┤ X ├─────┤ X ├─╫───■───╫────────■──┤ X ├»
                          └───┘     └───┘     └───┘ ║       ║           └───┘»
      bfocus_0: ────────────────────────────────────╫───────╫────────────────»
                                                    ║       ║                »
      bfocus_1: ────────────────────────────────────╫───────╫────────────────»
                                                    ║       ║                »
c_key_bfocus: 4/════════════════════════════════════╩═══════╩════════════════»
                                                    0       1                »
«                                              ┌───┐          
«     squares_0: ──────────────────────────────┤ X ├──────────
«                     ┌───┐                    └─┬─┘          
«     squares_1: ─────┤ X ├───────■──────────────┼────────────
«                     └─┬─┘       │              │            
«     squares_2: ──■────┼─────────┼──────────────┼────────────
«                  │    │         │              │            
«     squares_3: ──┼────┼─────────┼─────────■────┼────■───────
«                  │    │         │         │    │    │       
«         key_0: ──┼────┼─────────┼─────────┼────┼────┼───────
«                  │    │         │         │    │    │       
«         key_1: ──┼────┼─────────┼─────────┼────┼────┼───────
«                  │    │  ┌───┐  │         │    │    │  ┌───┐
«      afocus_0: ──┼────■──┤ X ├──┼─────────┼────■────┼──┤ X ├
«                  │    │  ├───┤  │  ┌───┐  │    │    │  ├───┤
«      afocus_1: ──┼────■──┤ X ├──┼──┤ X ├──┼────■────┼──┤ X ├
«                  │       └───┘┌─┴─┐└───┘┌─┴─┐ ┌─┐   │  └───┘
«      bfocus_0: ──┼────────────┤ X ├─────┤ X ├─┤M├───┼───────
«                ┌─┴─┐          └───┘     └───┘ └╥┘ ┌─┴─┐ ┌─┐ 
«      bfocus_1: ┤ X ├───────────────────────────╫──┤ X ├─┤M├─
«                └───┘                           ║  └───┘ └╥┘ 
«c_key_bfocus: 4/════════════════════════════════╩═════════╩══
«                                                2         3

#### Step 6

Lastly, run this circuit using `StatevectorSampler` and print out the result.

In [11]:
qc.draw()

┌───┐                                                        »
     squares_0: ┤ H ├────────────────────────────────────────────────────────»
                ├───┤                                                        »
     squares_1: ┤ H ├──■─────────────────────────────────────────────────────»
                ├───┤  │                                           ┌───┐     »
     squares_2: ┤ H ├──┼────■──────────────────────────────────────┤ X ├─────»
                ├───┤  │    │                         ┌───┐        └─┬─┘     »
     squares_3: ┤ H ├──┼────┼────■────■───────────────┤ X ├──────────┼───────»
                ├───┤  │    │    │    │            ┌─┐└─┬─┘          │       »
         key_0: ┤ H ├──┼────┼────┼────┼────■───────┤M├──┼────────────┼───────»
                ├───┤  │    │    │    │    │       └╥┘  │  ┌─┐       │       »
         key_1: ┤ H ├──┼────┼────┼────┼────┼────■───╫───┼──┤M├───────┼───────»
                └───┘┌─┴─┐  │  ┌─┴─┐  │  ┌─┴─┐  │   ║   │  └╥┘┌───┐  │  ┌───┐»
      afocus_0: ─────┤ X ├──┼──┤ X ├──┼──┤ X ├──┼───╫───■───╫─┤ X ├──■──┤ X ├»
                     └───┘┌─┴─┐└───┘┌─┴─┐└───┘┌─┴─┐ ║   │   ║ └───┘  │  ├───┤»
      afocus_1: ──────────┤ X ├─────┤ X ├─────┤ X ├─╫───■───╫────────■──┤ X ├»
                          └───┘     └───┘     └───┘ ║       ║           └───┘»
      bfocus_0: ────────────────────────────────────╫───────╫────────────────»
                                                    ║       ║                »
      bfocus_1: ────────────────────────────────────╫───────╫────────────────»
                                                    ║       ║                »
c_key_bfocus: 4/════════════════════════════════════╩═══════╩════════════════»
                                                    0       1                »
«                                              ┌───┐          
«     squares_0: ──────────────────────────────┤ X ├──────────
«                     ┌───┐                    └─┬─┘          
«     squares_1: ─────┤ X ├───────■──────────────┼────────────
«                     └─┬─┘       │              │            
«     squares_2: ──■────┼─────────┼──────────────┼────────────
«                  │    │         │              │            
«     squares_3: ──┼────┼─────────┼─────────■────┼────■───────
«                  │    │         │         │    │    │       
«         key_0: ──┼────┼─────────┼─────────┼────┼────┼───────
«                  │    │         │         │    │    │       
«         key_1: ──┼────┼─────────┼─────────┼────┼────┼───────
«                  │    │  ┌───┐  │         │    │    │  ┌───┐
«      afocus_0: ──┼────■──┤ X ├──┼─────────┼────■────┼──┤ X ├
«                  │    │  ├───┤  │  ┌───┐  │    │    │  ├───┤
«      afocus_1: ──┼────■──┤ X ├──┼──┤ X ├──┼────■────┼──┤ X ├
«                  │       └───┘┌─┴─┐└───┘┌─┴─┐ ┌─┐   │  └───┘
«      bfocus_0: ──┼────────────┤ X ├─────┤ X ├─┤M├───┼───────
«                ┌─┴─┐          └───┘     └───┘ └╥┘ ┌─┴─┐ ┌─┐ 
«      bfocus_1: ┤ X ├───────────────────────────╫──┤ X ├─┤M├─
«                └───┘                           ║  └───┘ └╥┘ 
«c_key_bfocus: 4/════════════════════════════════╩═════════╩══
«                                                2         3

In [12]:
## WRITE YOUR CODE BELOW HERE ##

sampler = StatevectorSampler()
pub = (qc)
job = sampler.run([pub], shots=1024)
result = job.result()       # Replace None with the result object from the sampler job

You should see in the ```c_key_bfocus``` classical register, the values of the first pair of qubits (key location) and the second pair of qubits (Bob's focus square) always match in each bitstring from the results.

In [13]:
result[0].data['c_key_bfocus']

BitArray(<shape=(), num_shots=1024, num_bits=4>)

In [14]:
# SUBMIT RESULT TO GRADER HERE (ex1c)
qsite24_grader_lab2ex1c(result)

Congratulations! 🎉 Your answer is correct.


### Part 2: Extension to a 16 square chessboard (4x4)

Now that you have solved the 4 square (2x2) problem, can you scale to a larger chessboard? For this next problem, find a solution to the problem when using a 4x4 chessboard. The rules are the same: Alice can only flip one coin on the board.

<center>
<table border=1>
    <colgroup>
        <col span="1" style="width:50px">
        <col span="1" style="width:50px">
        <col span="1" style="width:50px">
        <col span="1" style="width:50px">
    </colgroup>
    <tbody>
        <tr style="height:50px"><td></td> <td bgcolor='grey'> </td><td> </td><td bgcolor='grey'> </td></tr>
        <tr style="height:50px"><td bgcolor='grey'> </td><td></td><td bgcolor='grey'></td><td> </td></tr>
        <tr style="height:50px"><td> </td><td bgcolor='grey'> </td><td> </td><td bgcolor='grey'> </td></tr>
        <tr style="height:50px"><td bgcolor='grey'> </td><td> </td><td bgcolor='grey'> </td><td> </td></tr>
    </tbody>
</table>
</center>

Note that you'll need to use several more qubits. We suggest that you find a way to minimize the number of qubits in order to keep the memory requirements of the circuit simulation low enough to run on your computer. An initial solution we ran with 28 qubits used nearly 16 GB of RAM and took over 15 minutes to run. HINT: in your 2x2 solution, were there some qubits that you no longer needed at some point in your circuit and could reuse? If so, can you find a way in Qiskit to set them back to zero?

**IMPORTANT NOTE**: construct your circuit to place the final measurements in a classical register with the name ```c_key_focus``` that includes both the key location and Bob's focus square (similar to what was done in the 2x2 solution). Measure the key location in the lower half of the bits in this classical register, and measure Bob's focus square in the upper half of the bits. This is for format expected by the grader.

Simulate your circuit using the ```StatevectorSampler``` and submit the result object from the job object to the grader.

In [10]:
from qiskit import QuantumCircuit, ClassicalRegister, transpile, assemble
from qiskit_aer import Aer
# 定义比特数量
nb_coins = 16
nb_key = 4
nb_focus_first = 4
nb_focus_key = 4
nb_qubits = nb_coins + nb_key + nb_focus_first + nb_focus_key

# 添加经典寄存器
cr_key = ClassicalRegister(4, name='key')
cr_focus = ClassicalRegister(4, name='focus')
problem3_qc = QuantumCircuit(nb_qubits, name='Four-Square Chessboard')
problem3_qc.add_register(cr_key)
problem3_qc.add_register(cr_focus)

# 初始化棋盘上的硬币
for i in range(16):
    problem3_qc.h(i)

# 隐藏密钥
for i in range(16, 20):
    problem3_qc.h(i)

problem3_qc.barrier()

# 奇偶性计算，找到焦点初始位置并存储在20-23号比特
problem3_qc.cx(1, 20)
problem3_qc.cx(3, 20)
problem3_qc.cx(5, 20)
problem3_qc.cx(7, 20)
problem3_qc.cx(9, 20)
problem3_qc.cx(11, 20)
problem3_qc.cx(13, 20)
problem3_qc.cx(15, 20)

problem3_qc.cx(2, 21)
problem3_qc.cx(3, 21)
problem3_qc.cx(6, 21)
problem3_qc.cx(7, 21)
problem3_qc.cx(10, 21)
problem3_qc.cx(11, 21)
problem3_qc.cx(14, 21)
problem3_qc.cx(15, 21)

problem3_qc.cx(4, 22)
problem3_qc.cx(5, 22)
problem3_qc.cx(6, 22)
problem3_qc.cx(7, 22)
problem3_qc.cx(12, 22)
problem3_qc.cx(13, 22)
problem3_qc.cx(14, 22)
problem3_qc.cx(15, 22)

problem3_qc.cx(8, 23)
problem3_qc.cx(9, 23)
problem3_qc.cx(10, 23)
problem3_qc.cx(11, 23)
problem3_qc.cx(12, 23)
problem3_qc.cx(13, 23)
problem3_qc.cx(14, 23)
problem3_qc.cx(15, 23)

problem3_qc.barrier()

# 控制量子比特翻转棋盘硬币
ctrl_qubits = [20, 21, 22, 23]

# 翻转底行的硬币
problem3_qc.mcx(ctrl_qubits, 15)
problem3_qc.mcx(ctrl_qubits, 14)
problem3_qc.mcx(ctrl_qubits, 13)
problem3_qc.mcx(ctrl_qubits, 12)

# 翻转第三行的硬币
problem3_qc.mcx(ctrl_qubits, 11)
problem3_qc.mcx(ctrl_qubits, 10)
problem3_qc.mcx(ctrl_qubits, 9)
problem3_qc.mcx(ctrl_qubits, 8)

# 翻转第二行的硬币
problem3_qc.mcx(ctrl_qubits, 7)
problem3_qc.mcx(ctrl_qubits, 6)
problem3_qc.mcx(ctrl_qubits, 5)
problem3_qc.mcx(ctrl_qubits, 4)

# 翻转顶行的硬币
problem3_qc.mcx(ctrl_qubits, 3)
problem3_qc.mcx(ctrl_qubits, 2)
problem3_qc.mcx(ctrl_qubits, 1)
problem3_qc.mcx(ctrl_qubits, 0)

problem3_qc.barrier()

# 中间测量并重置16到19号比特，复用它们
problem3_qc.measure([16, 17, 18, 19], cr_key)
for qubit in range(16, 20):
    problem3_qc.reset(qubit)

# 焦点位置的最终奇偶性计算，复用之前的16到19号比特
problem3_qc.cx(1, 16)
problem3_qc.cx(3, 16)
problem3_qc.cx(5, 16)
problem3_qc.cx(7, 16)
problem3_qc.cx(9, 16)
problem3_qc.cx(11, 16)
problem3_qc.cx(13, 16)
problem3_qc.cx(15, 16)

problem3_qc.cx(2, 17)
problem3_qc.cx(3, 17)
problem3_qc.cx(6, 17)
problem3_qc.cx(7, 17)
problem3_qc.cx(10, 17)
problem3_qc.cx(11, 17)
problem3_qc.cx(14, 17)
problem3_qc.cx(15, 17)

problem3_qc.cx(4, 18)
problem3_qc.cx(5, 18)
problem3_qc.cx(6, 18)
problem3_qc.cx(7, 18)
problem3_qc.cx(12, 18)
problem3_qc.cx(13, 18)
problem3_qc.cx(14, 18)
problem3_qc.cx(15, 18)

problem3_qc.cx(8, 19)
problem3_qc.cx(9, 19)
problem3_qc.cx(10, 19)
problem3_qc.cx(11, 19)
problem3_qc.cx(12, 19)
problem3_qc.cx(13, 19)
problem3_qc.cx(14, 19)
problem3_qc.cx(15, 19)

# 再次测量焦点比特，准备下一步操作
problem3_qc.measure([16, 17, 18, 19], cr_focus)

# 使用QASM模拟器，它支持中间测量
simulator = Aer.get_backend('qasm_simulator')

# 编译电路以适应目标后端
compiled_circuit = transpile(problem3_qc, simulator)

# 执行电路
qobj = assemble(compiled_circuit, backend=simulator, shots=1024)
job = simulator.run(qobj)

# 获取结果
result = job.result()

# 提取测量的结果
counts = result.get_counts(compiled_circuit)
print(counts)


# 定义比特数量
nb_coins = 16
nb_key = 4
nb_focus_first = 4
nb_focus_key = 4
nb_aux = 1  # 添加一个辅助比特
nb_qubits = nb_coins + nb_key + nb_focus_first + nb_focus_key + nb_aux

# 添加经典寄存器
cr_key = ClassicalRegister(4, name='key')
cr_focus = ClassicalRegister(4, name='focus')
problem3_qc = QuantumCircuit(nb_qubits, name='Four-Square Chessboard')
problem3_qc.add_register(cr_key)
problem3_qc.add_register(cr_focus)

# 初始化棋盘上的硬币
for i in range(16):
    problem3_qc.h(i)

# 隐藏密钥
for i in range(16, 20):
    problem3_qc.h(i)

problem3_qc.barrier()

# 奇偶性计算，找到焦点初始位置并存储在20-23号比特
problem3_qc.cx(1, 20)
problem3_qc.cx(3, 20)
problem3_qc.cx(5, 20)
problem3_qc.cx(7, 20)
problem3_qc.cx(9, 20)
problem3_qc.cx(11, 20)
problem3_qc.cx(13, 20)
problem3_qc.cx(15, 20)

problem3_qc.cx(2, 21)
problem3_qc.cx(3, 21)
problem3_qc.cx(6, 21)
problem3_qc.cx(7, 21)
problem3_qc.cx(10, 21)
problem3_qc.cx(11, 21)
problem3_qc.cx(14, 21)
problem3_qc.cx(15, 21)

problem3_qc.cx(4, 22)
problem3_qc.cx(5, 22)
problem3_qc.cx(6, 22)
problem3_qc.cx(7, 22)
problem3_qc.cx(12, 22)
problem3_qc.cx(13, 22)
problem3_qc.cx(14, 22)
problem3_qc.cx(15, 22)

problem3_qc.cx(8, 23)
problem3_qc.cx(9, 23)
problem3_qc.cx(10, 23)
problem3_qc.cx(11, 23)
problem3_qc.cx(12, 23)
problem3_qc.cx(13, 23)
problem3_qc.cx(14, 23)
problem3_qc.cx(15, 23)

problem3_qc.barrier()

# 用辅助比特来优化 mcx 操作
ctrl_qubits = [20, 21, 22, 23]
aux_bit = 28  # 辅助比特的索引

# 第一阶段，使用辅助比特保存部分控制信息
problem3_qc.cx(20, aux_bit)  # 保存 20号比特的控制结果到辅助比特
problem3_qc.cx(21, aux_bit)  # 累加 21号比特

# 在第二阶段，将其他控制比特作用于目标
problem3_qc.mcx([aux_bit, 22, 23], 15)  # 使用22和23号比特进行操作

# 继续其他行的翻转
problem3_qc.mcx([aux_bit, 22, 23], 14)
problem3_qc.mcx([aux_bit, 22, 23], 13)
problem3_qc.mcx([aux_bit, 22, 23], 12)

problem3_qc.barrier()

# 中间测量并重置16到19号比特，复用它们
problem3_qc.measure([16, 17, 18, 19], cr_key)
for qubit in range(16, 20):
    problem3_qc.reset(qubit)

# 焦点位置的最终奇偶性计算，复用之前的16到19号比特
problem3_qc.cx(1, 16)
problem3_qc.cx(3, 16)
problem3_qc.cx(5, 16)
problem3_qc.cx(7, 16)
problem3_qc.cx(9, 16)
problem3_qc.cx(11, 16)
problem3_qc.cx(13, 16)
problem3_qc.cx(15, 16)

problem3_qc.cx(2, 17)
problem3_qc.cx(3, 17)
problem3_qc.cx(6, 17)
problem3_qc.cx(7, 17)
problem3_qc.cx(10, 17)
problem3_qc.cx(11, 17)
problem3_qc.cx(14, 17)
problem3_qc.cx(15, 17)

problem3_qc.cx(4, 18)
problem3_qc.cx(5, 18)
problem3_qc.cx(6, 18)
problem3_qc.cx(7, 18)
problem3_qc.cx(12, 18)
problem3_qc.cx(13, 18)
problem3_qc.cx(14, 18)
problem3_qc.cx(15, 18)

problem3_qc.cx(8, 19)
problem3_qc.cx(9, 19)
problem3_qc.cx(10, 19)
problem3_qc.cx(11, 19)
problem3_qc.cx(12, 19)
problem3_qc.cx(13, 19)
problem3_qc.cx(14, 19)
problem3_qc.cx(15, 19)

# 再次测量焦点比特，准备下一步操作
problem3_qc.measure([16, 17, 18, 19], cr_focus)

# 最终电路
problem3_qc.draw(output='mpl')

# 使用Qiskit的transpile工具优化电路
optimized_circuit = transpile(problem3_qc, optimization_level=3)
optimized_circuit.draw(output='mpl')


C:\Users\husky\AppData\Local\Temp\ipykernel_18796\693813704.py:147: DeprecationWarning: The function ``qiskit.compiler.assembler.assemble()`` is deprecated as of qiskit 1.2. It will be removed in the 2.0 release. The `Qobj` class and related functionality are part of the deprecated `BackendV1` workflow,  and no longer necessary for `BackendV2`. If a user workflow requires `Qobj` it likely relies on deprecated functionality and should be updated to use `BackendV2`.
  qobj = assemble(compiled_circuit, backend=simulator, shots=1024)


TypeError: 'QasmQobj' object is not iterable

In [9]:


# 检查是否能正确获取模拟器列表
print(Aer.backends())

[AerSimulator('aer_simulator'), AerSimulator('aer_simulator_statevector'), AerSimulator('aer_simulator_density_matrix'), AerSimulator('aer_simulator_stabilizer'), AerSimulator('aer_simulator_matrix_product_state'), AerSimulator('aer_simulator_extended_stabilizer'), AerSimulator('aer_simulator_unitary'), AerSimulator('aer_simulator_superop'), QasmSimulator('qasm_simulator'), StatevectorSimulator('statevector_simulator'), UnitarySimulator('unitary_simulator')]


In [ ]:
# 使用QASM模拟器，它支持中间测量
simulator = Aer.get_backend('qasm_simulator')

# 编译电路以适应目标后端
compiled_circuit = transpile(problem3_qc, simulator)

# 执行电路
qobj = assemble(compiled_circuit, backend=simulator, shots=1024)
job = simulator.run(qobj)

# 获取结果
result = job.result()

# 提取测量的结果
counts = result.get_counts(compiled_circuit)
print(counts)


In [1]:
from qiskit import QuantumCircuit, ClassicalRegister, transpile
from qiskit.primitives import StatevectorSampler

# 定义比特数量
nb_coins = 16
nb_key = 4
nb_focus_first = 4
nb_focus_key = 4
nb_qubits = nb_coins + nb_key + nb_focus_first + nb_focus_key

# 添加经典寄存器
cr_key = ClassicalRegister(4, name='key')
cr_focus = ClassicalRegister(4, name='focus')
qc = QuantumCircuit(nb_qubits, name='Four-Square Chessboard')
qc.add_register(cr_key)
qc.add_register(cr_focus)

# 初始化棋盘上的硬币
for i in range(16):
    qc.h(i)

# 隐藏密钥
for i in range(16, 20):
    qc.h(i)

qc.barrier()

# 奇偶性计算，找到焦点初始位置并存储在20-23号比特
# 使用循环简化生成 CNOT 门
for i in range(16):
    if i % 2 == 1:
        qc.cx(i, 20)
    if (i >> 1) % 2 == 1:
        qc.cx(i, 21)
    if (i >> 2) % 2 == 1:
        qc.cx(i, 22)
    if (i >> 3) % 2 == 1:
        qc.cx(i, 23)

qc.barrier()

# 控制量子比特翻转棋盘硬币
ctrl_qubits = [20, 21, 22, 23]

# 使用循环简化生成多控制量子门操作
for i in range(15, -1, -1):  # 从 15 到 0 的翻转操作
    qc.mcx(ctrl_qubits, i)

qc.barrier()

# 获取 q_reg_key 和 q_reg_bfocus 中所有量子比特
q_reg_key = [qc.qubits[i] for i in range(16, 20)]
q_reg_bfocus = [qc.qubits[i] for i in range(20, 24)]
all_qubits = q_reg_key + q_reg_bfocus

# 创建经典寄存器来存储测量结果
c_reg_key_bfocus = ClassicalRegister(8, name='key_bfocus')
qc.add_register(c_reg_key_bfocus)

# 对这些量子比特进行测量，结果存入 c_reg_key_bfocus
qc.measure(all_qubits, c_reg_key_bfocus)

# 优化电路
optimized_circuit = transpile(qc, optimization_level=3)



In [2]:
# 使用 StatevectorSampler
sampler = StatevectorSampler()
job = sampler.run([optimized_circuit], shots=1024)
result = job.result()

In [22]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorSampler

# Define quantum and classical registers
q_squares = QuantumRegister(16, name='squares')
q_key = QuantumRegister(4, name='key')
q_afocus = QuantumRegister(4, name='afocus')
q_bfocus = QuantumRegister(4, name='bfocus')
c_key_focus = ClassicalRegister(8, name='c_key_focus')

# Initialize the quantum circuit
qc = QuantumCircuit(q_squares, q_key, q_afocus, q_bfocus, c_key_focus)

# Coin distribution on each square (Hadamard gates)
for i in range(16):
    qc.h(q_squares[i])



# Hiding the key under one of the 16 squares (Hadamard gates on key qubits)
for i in range(4):
    qc.h(q_key[i])



# Parity check based on bit positions, placing results in q_afocus
parity_map = [
    (1, [1, 3, 5, 7, 9, 11, 13, 15]),
    (2, [2, 3, 6, 7, 10, 11, 14, 15]),
    (3, [4, 5, 6, 7, 12, 13, 14, 15]),
    (4, [8, 9, 10, 11, 12, 13, 14, 15])
]

for index, targets in enumerate(parity_map):
    for target in targets:
        qc.cx(q_squares[target], q_afocus[index])



# Adding modulo 2 of the position of the key and the position of the focus
for i in range(4):
    qc.cx(q_key[i], q_afocus[i])

ctrl_qubits = [q_afocus[i] for i in range(4)]



# Using mcx with control state '1111', '0111', etc., to turn the correct coins
states = ['1111', '0111', '1011', '0011', '1101', '0101', '1001', '0001', '1110', '0110', '1010', '0010', '1100', '0100', '1000', '0000']

for i, state in enumerate(states):
    qc.mcx(ctrl_qubits, q_squares[15 - i], ctrl_state=state)



# Parity check for q_bfocus based on bit positions
for index, targets in enumerate(parity_map):
    for target in targets:
        qc.cx(q_squares[target], q_bfocus[index])



# Measure q_key into the first 4 classical bits and q_bfocus into the next 4
qc.measure(q_key, c_key_focus[:4])
qc.measure(q_bfocus, c_key_focus[4:])


In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorSampler

# Define quantum and classical registers
q_squares = QuantumRegister(16, name='squares')
q_key = QuantumRegister(4, name='key')
q_afocus = QuantumRegister(4, name='afocus')
q_bfocus = QuantumRegister(4, name='bfocus')
c_key_focus = ClassicalRegister(8, name='c_key_focus')

# Initialize the quantum circuit
qc = QuantumCircuit(q_squares, q_key, q_afocus, q_bfocus, c_key_focus)

# Coin distribution on each square (Hadamard gates)

qc.h(q_squares)

# Hiding the key under one of the 16 squares (Hadamard gates on key qubits)
qc.h(q_key)



# Finding the parity of 1's on squares for which binary numbers finish by 1 and putting the answer on q_afocus[0]
qc.cx(q_squares[1], q_afocus[0])
qc.cx(q_squares[3], q_afocus[0])
qc.cx(q_squares[5], q_afocus[0])
qc.cx(q_squares[7], q_afocus[0])
qc.cx(q_squares[9], q_afocus[0])
qc.cx(q_squares[11], q_afocus[0])
qc.cx(q_squares[13], q_afocus[0])
qc.cx(q_squares[15], q_afocus[0])


# Finding the parity of 1's on squares for which binary numbers have a 1 as second-to-last digit and putting the answer on q_afocus[1]
qc.cx(q_squares[2], q_afocus[1])
qc.cx(q_squares[3], q_afocus[1])
qc.cx(q_squares[6], q_afocus[1])
qc.cx(q_squares[7], q_afocus[1])
qc.cx(q_squares[10], q_afocus[1])
qc.cx(q_squares[11], q_afocus[1])
qc.cx(q_squares[14], q_afocus[1])
qc.cx(q_squares[15], q_afocus[1])



# Finding the parity of 1's on squares for which binary numbers have a 1 as the 3rd digit from the right and putting the answer on q_afocus[2]
qc.cx(q_squares[4], q_afocus[2])
qc.cx(q_squares[5], q_afocus[2])
qc.cx(q_squares[6], q_afocus[2])
qc.cx(q_squares[7], q_afocus[2])
qc.cx(q_squares[12], q_afocus[2])
qc.cx(q_squares[13], q_afocus[2])
qc.cx(q_squares[14], q_afocus[2])
qc.cx(q_squares[15], q_afocus[2])



# Finding the parity of 1's on squares for which binary numbers have a 1 as the 4th digit from the right and putting the answer on q_afocus[3]
qc.cx(q_squares[8], q_afocus[3])
qc.cx(q_squares[9], q_afocus[3])
qc.cx(q_squares[10], q_afocus[3])
qc.cx(q_squares[11], q_afocus[3])
qc.cx(q_squares[12], q_afocus[3])
qc.cx(q_squares[13], q_afocus[3])
qc.cx(q_squares[14], q_afocus[3])
qc.cx(q_squares[15], q_afocus[3])



# Adding modulo 2 of the position of the key and the position of the focus
qc.cx(q_key[0], q_afocus[0])
qc.cx(q_key[1], q_afocus[1])
qc.cx(q_key[2], q_afocus[2])
qc.cx(q_key[3], q_afocus[3])



# Using mcx with control state '1111', '0111', etc., to turn the correct coins on each row
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[15], ctrl_state='1111')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[14], ctrl_state='0111')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[13], ctrl_state='1011')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[12], ctrl_state='0011')

qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[11], ctrl_state='1101')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[10], ctrl_state='0101')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[9], ctrl_state='1001')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[8], ctrl_state='0001')

qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[7], ctrl_state='1110')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[6], ctrl_state='0110')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[5], ctrl_state='1010')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[4], ctrl_state='0010')

qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[3], ctrl_state='1100')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[2], ctrl_state='0100')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[1], ctrl_state='1000')
qc.mcx([q_afocus[0], q_afocus[1], q_afocus[2], q_afocus[3]], q_squares[0], ctrl_state='0000')



# Finding the parity of 1's on squares for which binary numbers finish by 1 and putting the answer on q_bfocus[0]
qc.cx(q_squares[1], q_bfocus[0])
qc.cx(q_squares[3], q_bfocus[0])
qc.cx(q_squares[5], q_bfocus[0])
qc.cx(q_squares[7], q_bfocus[0])
qc.cx(q_squares[9], q_bfocus[0])
qc.cx(q_squares[11], q_bfocus[0])
qc.cx(q_squares[13], q_bfocus[0])
qc.cx(q_squares[15], q_bfocus[0])



# Finding the parity of 1's on squares for which binary numbers have a 1 as second-to-last digit and putting the answer on q_bfocus[1]
qc.cx(q_squares[2], q_bfocus[1])
qc.cx(q_squares[3], q_bfocus[1])
qc.cx(q_squares[6], q_bfocus[1])
qc.cx(q_squares[7], q_bfocus[1])
qc.cx(q_squares[10], q_bfocus[1])
qc.cx(q_squares[11], q_bfocus[1])
qc.cx(q_squares[14], q_bfocus[1])
qc.cx(q_squares[15], q_bfocus[1])



# Finding the parity of 1's on squares for which binary numbers have a 1 as the 3rd digit from the right and putting the answer on q_bfocus[2]
qc.cx(q_squares[4], q_bfocus[2])
qc.cx(q_squares[5], q_bfocus[2])
qc.cx(q_squares[6], q_bfocus[2])
qc.cx(q_squares[7], q_bfocus[2])
qc.cx(q_squares[12], q_bfocus[2])
qc.cx(q_squares[13], q_bfocus[2])
qc.cx(q_squares[14], q_bfocus[2])
qc.cx(q_squares[15], q_bfocus[2])



# Finding the parity of 1's on squares for which binary numbers have a 1 as the 4th digit from the right and putting the answer on q_bfocus[3]
qc.cx(q_squares[8], q_bfocus[3])
qc.cx(q_squares[9], q_bfocus[3])
qc.cx(q_squares[10], q_bfocus[3])
qc.cx(q_squares[11], q_bfocus[3])
qc.cx(q_squares[12], q_bfocus[3])
qc.cx(q_squares[13], q_bfocus[3])
qc.cx(q_squares[14], q_bfocus[3])
qc.cx(q_squares[15], q_bfocus[3])



# Measure q_key into the first 4 classical bits and q_bfocus into the next 4
qc.measure(q_key, c_key_focus[:4])
qc.measure(q_bfocus, c_key_focus[4:])


In [2]:
sampler = StatevectorSampler()
job = sampler.run([qc], shots=1024)
result = job.result()

In [26]:
print(result)

PrimitiveResult([SamplerPubResult(data=DataBin(c_key_focus=BitArray(<shape=(), num_shots=1024, num_bits=8>)), metadata={'shots': 1024, 'circuit_metadata': {}})], metadata={'version': 2})


In [25]:
import sys
import os
sys.path.append(os.path.abspath('D:/jupynote/Qsite/Q-SITE-IBM-Coding-Challenge/grader'))
from qsite24_ibm_grader import *

In [24]:
# SUBMIT MEASUREMENT COUNTS TO GRADER HERE (ex2)
qsite24_grader_lab2ex2(result)

Incorrect! The measured results are not as expected


### Congratulations!

You have completed this lab. We hope you enjoyed this experience and learned more about Qiskit!